# 09 · Advanced · Registry, run options, errors, and logging

The less common options: every `run()` argument, a private scanner registry,
handling configuration errors, and structured JSON logs.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.4.0-py3-none-any.whl

In [ ]:
import logging
from datetime import datetime, timezone

from lakehouse_gc import check, define_scanner, describe_rules, rules_text, run
from lakehouse_gc.core import PolicyValidationError, get_default_registry, loads_policy
from lakehouse_gc.core.log import JsonFormatter

POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"

# Every run() argument.
result = run(
    POLICY_PATH,  # path to policy.yaml; or policy=Policy(...), never both
    scanners=["uc"],  # names and/or scanner classes; None = all registered
    spark=spark,  # SparkSession, or any object with query(statement, params)
    run_id="manual-check-001",  # optional; default: random
    clock=lambda: datetime.now(timezone.utc),  # optional; must be timezone-aware
    # registry=...,  # optional (next cell)
)
print(result.run_id, result.started_at, result.finished_at)

# Rules as text or plain dicts (use rules_dataframe in notebooks).
print(rules_text(POLICY_PATH, scanners=["uc"]))
rule_rows = describe_rules(POLICY_PATH, scanners=["uc"])

In [ ]:
# Registry: the built-in scanners, plus a private copy with your own added.
print(get_default_registry().names())  # ('example', 'uc')


@check
def has_owner(item):
    return bool(item.get("owner"))


owners = define_scanner(
    "owners", resources=[{"name": "demo.a.b", "owner": "x"}], checks=[has_owner]
)
registry = get_default_registry().copy()
registry.register(owners)
with_registry = run(POLICY_PATH, scanners=["uc", "owners"], spark=spark, registry=registry)
print(with_registry.scanners)
# To add a scanner for the whole session instead:
#   from lakehouse_gc.core import register_scanner; register_scanner(owners)

In [ ]:
# Configuration mistakes fail before anything is scanned, naming the exact key.
try:
    loads_policy("policy_version: 1\nsources:\n  tabels: org.meta.tables\n")
except PolicyValidationError as error:
    print(error.path, "->", error.message)  # sources.tabels -> unknown source (did you mean ...

In [ ]:
# Structured JSON logs (fields: event, run_id, scanner, phase, status, ...).
json_handler = logging.StreamHandler()
json_handler.setFormatter(JsonFormatter())
logger = logging.getLogger("lakehouse_gc")
logger.addHandler(json_handler)
logger.setLevel(logging.INFO)
run(POLICY_PATH, scanners=["uc"], spark=spark)
logger.removeHandler(json_handler)